In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [4]:
tensorflow.random.set_seed(35) #for the reproducibility of results

In [14]:
def design_model(features):
  model = Sequential(name = "my_model")
  #without hard-coding
  input = InputLayer(shape=(features.shape[1],)) 
  #add the input layer
  model.add(input) 
  #add a hidden layer with 128 neurons
  model.add(Dense(128, activation='relu')) 
  #add an output layer to our model
  model.add(Dense(1)) 
  opt = Adam(learning_rate=0.1)
  model.compile(loss='mse',  metrics=['mae'], optimizer=opt)
  return model

In [7]:
ds = pd.read_csv('insurance.csv') #load the dataset
ds.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [8]:
features = ds.iloc[:,0:6] #choose first 7 columns as features
labels = ds.charges # choose the final column for prediction

In [9]:
features = pd.get_dummies(features) #one-hot encoding for categorical variables
features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.33, random_state=42)

In [10]:
# standardize
ct = ColumnTransformer([('standardize', StandardScaler(), ['age', 'bmi', 'children'])], remainder='passthrough')
features_train = ct.fit_transform(features_train)
features_test = ct.transform(features_test)

In [15]:
# invoke the function for our model design
model = design_model(features_train)
model.summary()

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 128)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,665 (6.50 KB)

 Trainable params: 1,665 (6.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# fit the model using 40 epochs and batch size 1
model.fit(features_train, labels_train, epochs=40, batch_size=1, verbose=1)

Epoch 1/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 64754700.0000 - mae: 5172.2642 
Epoch 2/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 29725974.0000 - mae: 3378.6914
Epoch 3/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 27719520.0000 - mae: 3186.7188
Epoch 4/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 27215396.0000 - mae: 3139.9202
Epoch 5/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 27028934.0000 - mae: 3122.1272
Epoch 6/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 26913732.0000 - mae: 3113.1218
Epoch 7/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 26802836.0000 - mae: 3104.6770
Epoch 8/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 26712336.0000 - mae: 3097.7910
Epoch 9/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 26636036.0000 - mae: 3092.7476
Epoch 10/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 26556130.0000 - mae: 3087.1292
Epoch 11/40
896/896 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 26485646.0000 - mae: 3081.9

In [17]:
# evaluate the model on the test data
val_mse, val_mae = model.evaluate(features_test, labels_test, verbose=0)
val_mae

2256.906494140625